In [1]:
import pickle
import pandas as pd
from datetime import datetime
import numpy as np
import os

### Clean Data and make speech and income data compatible

In [2]:
speech_data = pickle.load(open("data/pp20_coded.pkl","rb"))

speech_data.loc[speech_data.fraction_or_role == "Die Linke", "fraction_or_role"] = "DIE LINKE"

old_topics = speech_data.topic.unique()
old_topics.sort()

new_topics = ["Labor", "Education", "Civil Rights", "Energy", "Healthcare", "Infrastructure", "Domestic Governance", "International Affairs",
             "Culture", "Agriculture", "Macroeconomy", "Migration", "Parliamentary Affairs", "Law and Order", "Social Welfare", "Government Operations",
             "Technology and Media", "Environment", "Defense", "Housing"]

new_names = speech_data.topic.values
for old_name, new_name in zip(old_topics, new_topics):
    new_names = [s.replace(old_name, new_name) for s in new_names]

speech_data.topic = new_names

date_year = [int(date[-4:]) for date in speech_data.date]
speech_data.insert(2, "date_year", date_year)

speech_data.head()

,document_name,date,date_year,top_id,drucksachen,context,speaker_id,first_name,last_name,full_name,fraction_or_role,speech_id,text,topic
21,20100,27.04.2023,2023,Tagesordnungspunkt 21,"Drucksachen 20/4675, 20/6521","Wärmewende versorgungssicher, nachhaltig und s...",11005083,Bernhard,Herrmann,Bernhard Herrmann,BÜNDNIS 90/DIE GRÜNEN,ID2010002000,Sehr geehrte Frau Präsidentin! Liebe Kolleginn...,Energy
51,20100,27.04.2023,2023,Zusatzpunkt 3 und 4,Drucksache 20/6544; Drucksache 20/6546,Gute Pflege stabil finanzieren,11004829,Axel,Müller,Axel Müller,CDU/CSU,ID2010004900,Sehr geehrte Frau Präsidentin! Sehr geehrter M...,Healthcare
72,20100,27.04.2023,2023,Tagesordnungspunkt 12,Drucksache 20/6520,,11004023,Marco,Buschmann,Marco Buschmann,Bundesminister der Justiz,ID2010006900,Sehr geehrte Frau Präsidentin! Liebe Kolleginn...,Law and Order
79,20100,27.04.2023,2023,Tagesordnungspunkt 12,Drucksache 20/6520,,11005078,Linda,Heitmann,Linda Heitmann,BÜNDNIS 90/DIE GRÜNEN,ID2010007500,Frau Präsidentin! Liebe Kolleginnen und Kolleg...,Domestic Governance
83,20100,27.04.2023,2023,Tagesordnungspunkt 11,"Drucksachen 20/4310, 20/6481",Straßenblockierer und Museumsrandalierer härte...,11004891,Thomas,Seitz,Thomas Seitz,AfD,ID2010007900,Sehr geehrte Frau Präsidentin! Sehr geehrte Da...,Civil Rights


In [ ]:
try:
    income_data = pickle.load(open("/home/til/Documents/python/powi/german-mps-moonlighting/data/pp20_income_data.pkl", "rb"))
except:
    income_data = pickle.load(open("data/pp20_income_data.pkl", "rb"))

income_name = ["Matthias Mieves", "Isabel Cademartori", "Rainer Keller", "Inge Gräßle", "Swantje Michaelsen", "Gero Hocker", "Jamila Anna Schäfer", "Max Mordhorst",
               "Johann Wadephul", "Christian von Stetten", "Gyde Jensen-Bornhöft", "Julian Simon Grünke", "Michael Link", "Anne-Monika Spallek", "Adis Ahmetović", "Bettina Wiesmann",
               "Reem Alabali Radovan"]

speech_name = ["Matthias David Mieves", "Isabel Cademartori Dujisin", "Rainer Johannes Keller", "Ingeborg Gräßle", "Swantje Henrike Michaelsen", "Gero Clemens Hocker", "Jamila Schäfer", "Maximilian Mordhorst",
               "Johann David Wadephul", "Christian Freiherr von Stetten", "Gyde Jensen", "Julian Grünke", "Michael Georg Link", "Anne Monika Spallek", "Adis Ahmetovic", "Bettina Margarethe Wiesmann",
               "Reem Alabali-Radovan"]

if len(income_name) != len(speech_name):
    print("arrays have different length")

new_names = income_data.name.values
for old_name, new_name in zip(income_name, speech_name):
    new_names = [s.replace(old_name, new_name) for s in new_names]

income_data.name = new_names

in_fraction_since_split = income_data.fraction.str.split(" seit ", expand=True).rename(columns={0: "fraction", 1: "in_fraction_since"})
income_data.fraction = in_fraction_since_split.fraction.values
income_data.insert(3, "in_fraction_since", in_fraction_since_split.in_fraction_since)

income_data.loc[income_data.fraction == "Die Linke. (Gruppe)", "fraction"] = "DIE LINKE"
income_data.loc[income_data.fraction == "BSW (Gruppe)", "fraction"] = "BSW"
income_data.loc[income_data.fraction == "fraktionslos", "fraction"] = "Fraktionslos"


income_data.head()

,mandates_id,name,fraction,in_fraction_since,parliament_period,created,data_change_date,job_label,job_category,job_organisation,job_topic,interval,income_level,income
77,53449,Fritz Güntzler,CDU/CSU,None,Bundestag 2021 - 2025,1750174491,2025-06-17,Übernahme von Reisekosten,29232,Internationales Steuerseminar Schweiz (IStS),Finanzen,NaN,2,4411.51
104,53647,Stefan Schmidt,BÜNDNIS 90/DIE GRÜNEN,None,Bundestag 2021 - 2025,1749628691,2025-06-11,Mitglied des Zukunftsrates (ab November 2024),29230,Verband der Sparda-Banken e.V.,"Öffentliche Finanzen, Steuern und Abgaben",NaN,1,2000.00
114,54131,Carsten Linnemann,CDU/CSU,None,Bundestag 2021 - 2025,1748854863,2025-06-02,Generalsekretär,29647,Christlich Demokratische Union Deutschlands,"Politisches Leben, Parteien",1,3,11227.20
120,54027,Johannes Vogel,FDP,None,Bundestag 2021 - 2025,1744095785,2025-04-08,Übernahme Reisekosten,29232,Global Public Policy Institute,Außenpolitik und internationale Beziehungen,NaN,3,10955.00
121,53525,Rasha Nasr,SPD,None,Bundestag 2021 - 2025,1744095631,2025-04-08,Übernahme von Reisekosten,29232,Robert Bosch Stiftung GmbH,"Gesellschaftspolitik, soziale Gruppen",NaN,3,8314.00


In [5]:
job_topic_mapping = {
    'Finanzen': 'Macroeconomy',
    'Öffentliche Finanzen, Steuern und Abgaben': 'Macroeconomy',
    'Politisches Leben, Parteien': 'Government Operations',
    'Außenpolitik und internationale Beziehungen': 'International Affairs',
    'Gesellschaftspolitik, soziale Gruppen': 'Culture',
    'Europapolitik und Europäische Union': 'International Affairs',
    'Energie': 'Energy',
    'Recht': 'Law and Order',
    'Wirtschaft': 'Macroeconomy',
    'Staat und Verwaltung': 'Government Operations',
    'Landwirtschaft und Ernährung': 'Agriculture',
    'Gesundheit': 'Healthcare',
    'Medien': 'Technology and Media',
    'Kultur': 'Culture',
    'Bildung und Erziehung': 'Education',
    'Verkehr': 'Infrastructure',
    'Umwelt': 'Environment',
    'Raumordnung, Bau- und Wohnungswesen': 'Housing',
    'Jugend': 'Education',
    'Sport': 'Domestic Governance',
    'Entwicklungspolitik': 'International Affairs',
    'Forschung': 'Technology and Media',
    'Außenwirtschaft': 'Macroeconomy',
    'Medien, Kommunikation und Informationstechnik': 'Technology and Media',
    'Soziale Sicherung': 'Social Welfare',
    'Sport, Freizeit und Tourismus': 'Domestic Governance',
    'digitale Infrastruktur': 'Infrastructure',
    'Verbraucherschutz': 'Domestic Governance',
    'Tourismus': 'Domestic Governance',
    'Wissenschaft, Forschung und Technologie': 'Technology and Media',
    'Verteidigung': 'Defense',
    'Arbeit und Beschäftigung': 'Labor', 
    'Innere Sicherheit': 'Law and Order',
    'Menschenrechte': 'International Affairs',
    'Reaktorsicherheit': 'Environment',
    'Digitale Agenda': 'Technology and Media',
    'Migration und Aufenthaltsrecht': 'Migration',
    'Bundestag': 'Government Operations',
    'Innere Angelegenheiten': 'Domestic Governance',
    'Klima': 'Environment',
    'Naturschutz': 'Environment'
}

job_topic_mapping_de = {
    'Finanzen': 'Makroökonomie',
    'Öffentliche Finanzen, Steuern und Abgaben': 'Makroökonomie',
    'Politisches Leben, Parteien': 'Staatliche Aufgaben und Öffentliche Versorgung',
    'Außenpolitik und internationale Beziehungen': 'Internationale Beziehungen',
    'Gesellschaftspolitik, soziale Gruppen': 'Kultur und Gesellschaftspolitik',
    'Europapolitik und Europäische Union': 'Internationale Beziehungen',
    'Energie': 'Energie',
    'Recht': 'Recht und Ordnung',
    'Wirtschaft': 'Makroökonomie',
    'Staat und Verwaltung': 'Staatliche Aufgaben und Öffentliche Versorgung',
    'Landwirtschaft und Ernährung': 'Landwirtschaft',
    'Gesundheit': 'Gesundheit',
    'Medien': 'Technologie und Medien',
    'Kultur': 'Kultur und Gesellschaftspolitik',
    'Bildung und Erziehung': 'Bildung',
    'Verkehr': 'Infrastruktur',
    'Umwelt': 'Umwelt',
    'Raumordnung, Bau- und Wohnungswesen': 'Wohnungswesen',
    'Jugend': 'Bildung',
    'Sport': 'Innenpolitik',
    'Entwicklungspolitik': 'Internationale Beziehungen',
    'Forschung': 'Technologie und Medien',
    'Außenwirtschaft': 'Makroökonomie',
    'Medien, Kommunikation und Informationstechnik': 'Technologie und Medien',
    'Soziale Sicherung': 'Sozialpolitik',
    'Sport, Freizeit und Tourismus': 'Innenpolitik',
    'digitale Infrastruktur': 'Infrastruktur',
    'Verbraucherschutz': 'Innenpolitik',
    'Tourismus': 'Innenpolitik',
    'Wissenschaft, Forschung und Technologie': 'Technologie und Medien',
    'Verteidigung': 'Verteidigung',
    'Arbeit und Beschäftigung': 'Arbeit', 
    'Innere Sicherheit': 'Recht und Ordnung',
    'Menschenrechte': 'Internationale Beziehungen',
    'Reaktorsicherheit': 'Umwelt',
    'Digitale Agenda': 'Technologie und Medien',
    'Migration und Aufenthaltsrecht': 'Migration',
    'Bundestag': 'Staatliche Aufgaben und Öffentliche Versorgung',
    'Innere Angelegenheiten': 'Innenpolitik',
    'Klima': 'Umwelt',
    'Naturschutz': 'Umwelt'
}

new_job_topics = [job_topic_mapping[jt] if not pd.isna(jt) else jt for jt in income_data.job_topic.values]
income_data.job_topic = new_job_topics

created_year = [datetime.fromtimestamp(timestamp).year for timestamp in income_data.created]
income_data.insert(6, "created_year", created_year)

income_data.to_csv("data/pp20_income_data_compatible.csv", index=False)

income_data.head()

,mandates_id,name,fraction,in_fraction_since,parliament_period,created,created_year,data_change_date,job_label,job_category,job_organisation,job_topic,interval,income_level,income
77,53449,Fritz Güntzler,CDU/CSU,None,Bundestag 2021 - 2025,1750174491,2025,2025-06-17,Übernahme von Reisekosten,29232,Internationales Steuerseminar Schweiz (IStS),Macroeconomy,NaN,2,4411.51
104,53647,Stefan Schmidt,BÜNDNIS 90/DIE GRÜNEN,None,Bundestag 2021 - 2025,1749628691,2025,2025-06-11,Mitglied des Zukunftsrates (ab November 2024),29230,Verband der Sparda-Banken e.V.,Macroeconomy,NaN,1,2000.00
114,54131,Carsten Linnemann,CDU/CSU,None,Bundestag 2021 - 2025,1748854863,2025,2025-06-02,Generalsekretär,29647,Christlich Demokratische Union Deutschlands,Government Operations,1,3,11227.20
120,54027,Johannes Vogel,FDP,None,Bundestag 2021 - 2025,1744095785,2025,2025-04-08,Übernahme Reisekosten,29232,Global Public Policy Institute,International Affairs,NaN,3,10955.00
121,53525,Rasha Nasr,SPD,None,Bundestag 2021 - 2025,1744095631,2025,2025-04-08,Übernahme von Reisekosten,29232,Robert Bosch Stiftung GmbH,Culture,NaN,3,8314.00


### Convert Data into Panel Format

In [6]:
mp_year_topic_dict = {
    "mandates_id": [],
    "mp": [],
    "year": [],
    "fraction_or_role": [],
    "topic": [],
    "topic_total_count": [],
    "topic_share": [],
    "earnings": []
}

speech_group_by_name = speech_data.groupby("full_name")
speech_names = list(speech_group_by_name.groups.keys())

income_group_by_name = income_data.groupby("name")

years = speech_data.date_year.unique()
years.sort()

for name in speech_names:
    speech_snippet = speech_group_by_name.get_group(name).sort_values(by="date_year")
    fraction_or_role = speech_snippet.fraction_or_role.values[0]
    try:
        income_snippet = income_group_by_name.get_group(name)
        mandates_id = income_snippet.mandates_id.unique()[0]
        mp_has_income = True
    except:
        mandates_id = None
        mp_has_income = False

    for y in years:
        year_snippet = speech_snippet[speech_snippet.date_year == y]
        year_shape = year_snippet.shape[0]

        try:        
            fraction_or_role = year_snippet.fraction_or_role.unique()[0]
        except:
            pass

        topics = year_snippet.topic.value_counts().index.to_list()
        topic_total_count = [count for count in year_snippet.topic.value_counts().to_list()]
        topic_shares = [count/year_shape for count in year_snippet.topic.value_counts().to_list()]

        missing_topics = [t for t in new_topics if t not in topics]
        missing_len = len(missing_topics)
        topics.extend(missing_topics)
        topic_total_count.extend([float('nan') for k in range(missing_len)])
        topic_shares.extend([float('nan') for k in range(missing_len)])

        if mp_has_income:
            earnings = []
            income_year_snippet = income_snippet[income_snippet.created_year == y]
            for t in topics:
                income_topic_snippet = income_year_snippet[income_year_snippet.job_topic == t]
                topic_earnings = income_topic_snippet.income.mean()
                earnings.append(topic_earnings)
        else:
            earnings = [float('nan') for k in range(len(topics))]
        
        mp_year_topic_dict["mandates_id"].extend([mandates_id for k in range(len(topics))])
        mp_year_topic_dict["mp"].extend([name for k in range(len(topics))])
        mp_year_topic_dict["year"].extend([y for k in range(len(topics))])
        mp_year_topic_dict["fraction_or_role"].extend([fraction_or_role for k in range(len(topics))])
        mp_year_topic_dict["topic"].extend(topics)
        mp_year_topic_dict["topic_total_count"].extend(topic_total_count)
        mp_year_topic_dict["topic_share"].extend(topic_shares)
        mp_year_topic_dict["earnings"].extend(earnings)

In [ ]:
save = False

mp_year_topic_df = pd.DataFrame(mp_year_topic_dict)

mp_id_dict = {mp: 10001 + i for i, mp in enumerate(mp_year_topic_df.mp.unique())}
mp_ids = [mp_id_dict[mp] for mp in mp_year_topic_df.mp.values]

mp_year_topic_df.insert(0, "id", mp_ids)

mp_year_topic_df = mp_year_topic_df.sort_values(by=["mp", "topic", "year"])
mp_year_topic_df = mp_year_topic_df.fillna(0)
mp_year_topic_df.mandates_id = mp_year_topic_df.mandates_id.astype(int)
mp_year_topic_df.topic_total_count = mp_year_topic_df.topic_total_count.astype(int)

mp_times_topic = []
for i in range(len(mp_year_topic_df.mp.unique()) * len(mp_year_topic_df.topic.unique())):
     mp_times_topic.extend([i+1 for k in range(5)])
mp_year_topic_df.insert(6, "mp_times_topic", mp_times_topic)

mp_year_topic_df = mp_year_topic_df.reset_index(drop=True)

######  Vary who is considered treated based on different income levels.

mp_year_topic_df["earnings_above_1000"] = [0 if earnings <= 1000 else earnings for earnings in mp_year_topic_df.earnings.values]
mp_year_topic_df["earnings_above_5000"] = [0 if earnings <= 5000 else earnings for earnings in mp_year_topic_df.earnings.values]
mp_year_topic_df["earnings_above_10000"] = [0 if earnings <= 10000 else earnings for earnings in mp_year_topic_df.earnings.values]
mp_year_topic_df["earnings_above_50000"] = [0 if earnings <= 50000 else earnings for earnings in mp_year_topic_df.earnings.values]

###### Add Dummys for years before and after job entry.

mp_group = mp_year_topic_df.groupby("mp") 
mp_keys = list(mp_group.groups.keys())

job_entry_minus_1 = []
job_entry_minus_2 = []
job_entry_plus_1 = []
job_entry_plus_2 = []

for mp_key in mp_keys:
    get_mp = mp_group.get_group(mp_key)
    mp_topic_group = get_mp.groupby("topic")
    mp_topic_group_keys = list(mp_topic_group.groups.keys())

    job_entry_minus_1_mp = []
    job_entry_minus_2_mp = []
    job_entry_plus_1_mp = []
    job_entry_plus_2_mp = []

    for topic_key in mp_topic_group_keys:
        g = mp_topic_group.get_group(topic_key)

        job_entry_minus_1_topic = [0 for i in range(5)]
        job_entry_minus_2_topic = [0 for i in range(5)]
        job_entry_plus_1_topic = [0 for i in range(5)]
        job_entry_plus_2_topic = [0 for i in range(5)]

        earnings_index = np.nonzero(g.earnings.values)[0]
        for k, ei in enumerate(earnings_index):
                year_before = ei - 1
                if year_before not in earnings_index and year_before >= 0:
                    job_entry_minus_1_topic[year_before] = 1

                if k == 0:
                    year_before = ei - 2
                    if year_before not in earnings_index and year_before >= 0:
                        job_entry_minus_2_topic[year_before] = 1
                elif k > 0:
                    if ei - earnings_index[k-1] != 1:
                        year_before = ei - 2
                        if year_before not in earnings_index and year_before >= 0:
                            job_entry_minus_2_topic[year_before] = 1
                
                year_after = ei + 1
                if year_after not in earnings_index and year_after <= 4:
                    job_entry_plus_1_topic[year_after] = 1
            
                if k == len(earnings_index)-1:
                    year_after = ei + 2
                    if year_after not in earnings_index and year_after <= 4:
                        job_entry_plus_2_topic[year_after] = 1
                elif k < len(earnings_index)-1:
                    if earnings_index[k+1] - ei != 1:
                        year_after = ei + 2
                        if year_after not in earnings_index and year_after <= 4:
                            job_entry_plus_2_topic[year_after] = 1
        
        job_entry_minus_1_mp.extend(job_entry_minus_1_topic)
        job_entry_minus_2_mp.extend(job_entry_minus_2_topic)
        job_entry_plus_1_mp.extend(job_entry_plus_1_topic)
        job_entry_plus_2_mp.extend(job_entry_plus_2_topic)

    job_entry_minus_1.extend(job_entry_minus_1_mp)
    job_entry_minus_2.extend(job_entry_minus_2_mp)
    job_entry_plus_1.extend(job_entry_plus_1_mp)
    job_entry_plus_2.extend(job_entry_plus_2_mp)     

mp_year_topic_df["in_job"] = [0 if earnings == 0 else 1 for earnings in mp_year_topic_df.earnings.values]
mp_year_topic_df["job_entry_minus_1"] = job_entry_minus_1
mp_year_topic_df["job_entry_minus_2"] = job_entry_minus_2
mp_year_topic_df["job_entry_plus_1"] = job_entry_plus_1
mp_year_topic_df["job_entry_plus_2"] = job_entry_plus_2

######

if save:
     if not os.path.isdir("data/"):
         os.mkdir("data/")
     mp_year_topic_df.to_csv("data/mp_year_topic_df.csv", index=False)

mp_year_topic_df

,id,mandates_id,mp,year,fraction_or_role,topic,mp_times_topic,topic_total_count,topic_share,earnings,earnings_above_1000,earnings_above_5000,earnings_above_10000,earnings_above_50000,in_job,job_entry_minus_1,job_entry_minus_2,job_entry_plus_1,job_entry_plus_2
0,10001,53839,Achim Post,2021,SPD,Agriculture,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0
1,10001,53839,Achim Post,2022,SPD,Agriculture,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0
2,10001,53839,Achim Post,2023,SPD,Agriculture,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0
3,10001,53839,Achim Post,2024,SPD,Agriculture,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0
4,10001,53839,Achim Post,2025,SPD,Agriculture,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80795,10808,0,Zoe Mayer,2021,BÜNDNIS 90/DIE GRÜNEN,Technology and Media,16160,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0
80796,10808,0,Zoe Mayer,2022,BÜNDNIS 90/DIE GRÜNEN,Technology and Media,16160,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0
80797,10808,0,Zoe Mayer,2023,BÜNDNIS 90/DIE GRÜNEN,Technology and Media,16160,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0
80798,10808,0,Zoe Mayer,2024,BÜNDNIS 90/DIE GRÜNEN,Technology and Media,16160,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0
